# 03 — Disentanglement measurement

The headline: does β actually disentangle, measured (MIG/DCI) against the
fantasy-map factor oracle? Loads the committed Stage 3 results and **asserts**
the CLAIMS.md numbers, so running this notebook is a self-check. Metric
estimators are calibration-gated (`uv run pytest -m gate_stage3`).

In [ ]:
import json
from pathlib import Path
R = Path('..') / 'results'
metrics = json.loads((R/'stage3_metrics.json').read_text())
verdicts = json.loads((R/'stage3_verdicts.json').read_text())
print('factors:', metrics['factors'])
print('rows:', len(metrics['results']))

## Headline figure
MIG vs β (left, with the PCA baseline) and reconstruction vs β (right).

In [ ]:
from IPython.display import Image
Image(filename=str(R/'headline.png'))

## H1–H3 verdicts (64px headline, Holm-corrected α=0.05)

In [ ]:
v = verdicts['64']
for h in ('H1','H2','H3'):
    d=v[h]; print(f"{h}: {'SUPPORTED' if d['reject'] else 'not supported'}  p_holm={d['p_holm']:.4f}")
print('MIG by beta:', {b:round(m[0],3) for b,m in v['mig_by_beta'].items()})
print('recon by beta:', {b:round(m[0],4) for b,m in v['recon_by_beta'].items()})

## Assertions — these encode CLAIMS.md (C1–C6)

In [ ]:
v64, v128 = verdicts['64'], verdicts['128']
# C1 (H1): MIG does NOT rise significantly with beta at 64px
assert not v64['H1']['reject'], 'C1: H1 should not be supported'
# C2 (H2): rate-distortion tradeoff holds at 64px
assert v64['H2']['reject'] and v64['H2']['spearman'] > 0.9, 'C2: H2 should be supported'
# C3 (H3): best beta-VAE beats PCA on MIG at 64px
assert v64['H3']['reject'] and v64['H3']['best_mig'] > v64['H3']['pca_mig'], 'C3: H3 should hold'
# C4: absolute MIG is low across the sweep
migs=[r['mig'] for r in metrics['results'] if r['model']=='bvae']
assert max(migs) < 0.15, 'C4: MIG should be low everywhere'
# C5: PCA DCI-disentanglement exceeds every beta-VAE cell
pca_d=[r['dci_disentanglement'] for r in metrics['results'] if r['model']=='pca']
vae_d=[r['dci_disentanglement'] for r in metrics['results'] if r['model']=='bvae']
assert min(pca_d) > max(vae_d), 'C5: PCA DCI-D should top the VAEs'
# C6: at 128px the beta effect vanishes (H1,H2 not supported; H3 holds)
assert not v128['H1']['reject'] and not v128['H2']['reject'] and v128['H3']['reject'], 'C6'
print('all CLAIMS assertions pass ✓')

## Reading — Locatello et al. (2019)

A **replication-skeptical** result. Increasing β does not reliably increase
disentanglement (H1 fails at both resolutions; across-seed bands overlap), even
though β mechanically trades reconstruction for compression on the rate–distortion
frontier (H2, 64px). The β-VAE beats a linear PCA baseline on MIG (H3) but only at a
low absolute level (MIG≈0.1), and MIG vs DCI disagree on whether it beats PCA at all
(PCA wins on DCI-disentanglement). Seed- and metric-dependence of the disentanglement
verdict is exactly what the impossibility result predicts for the unsupervised setting.